# 01 · Caso de negocio — RiskVision

**Grupo 6 (Analistas de Datos):** Juan David Ardila Arboleda, Lina Isabel Vasquez Villegas, Andres Felipe Orozco Roa, Cristina Garces y Estefania Cervera.
**Dataset:** *Default of Credit Card Clients* (UCI Machine Learning Repository, licencia CC BY 4.0), 30.000 clientes de un banco de Taiwán.

## Para qué sirve este notebook
El proyecto necesita tres cosas escritas con números: una pregunta de negocio, una métrica que la refleje y una hipótesis de cuánto cuesta equivocarse. Este notebook las define y su resumen va a `reports/00_caso.md`.

Una aclaración honesta: cuando se escribió, el grupo ya había explorado los datos y probado un modelo en otro notebook. Por eso se conservan algunos resultados previos, marcados como exploratorios.

## Cómo leerlo
Cada sección responde una pregunta. Las celdas de código hacen cuentas cortas y debajo de cada una hay una **lectura** en palabras simples.

## Palabras clave
| Término | Significado simple |
|---|---|
| **Falso negativo** | El modelo dice «paga» y el cliente no paga. Se pierde la oportunidad de revisarlo a tiempo. Suponemos que es el error más caro |
| **Falso positivo** | El modelo dice «incumple» y el cliente sí pagaba. Genera una revisión innecesaria |
| **Matriz de costos** | Tabla que dice cuánto pesa cada tipo de error, en unidades relativas |
| **Línea base** | Resultado de una estrategia sin modelo. El modelo solo vale la pena si la supera |
| **Umbral** | Nivel de riesgo desde el cual se marca a un cliente |


## 1. La pregunta de negocio

**Pregunta de negocio:** ¿Qué clientes de tarjeta de crédito tienen alta probabilidad de no pagar el próximo mes?

**¿Quién decide y con qué información?** El área de riesgo de crédito del banco. Cuenta con el límite de crédito, datos demográficos, el estado de pago de los últimos 6 meses y los montos facturados y pagados. Con la probabilidad de impago de cada cliente decide si ajusta el límite, hace seguimiento preventivo u ofrece reestructurar la deuda.

**¿Qué pasa hoy, sin el modelo?** No hay un orden de prioridad por riesgo: el equipo revisa a los clientes con reglas simples o sin un criterio común, y no sabe a quién contactar primero. *(Es una descripción general: los datos no dicen cómo trabaja hoy el banco y el grupo debe confirmarlo.)*


## 2. Equivocarse no cuesta lo mismo

**El error más caro (según nuestra hipótesis):** no detectar a un cliente que sí va a incumplir (falso negativo). Se pierde la oportunidad de revisar su caso a tiempo. Los datos no permiten medir cuánto dinero se pierde, ni suponer que equivale a todo lo prestado.

**El error más barato:** marcar como riesgoso a un cliente que sí iba a pagar (falso positivo). Aquí lo tratamos como una revisión preventiva innecesaria. Ajustar un límite o reestructurar una deuda son decisiones distintas, y sus costos no están medidos.

**De dónde sale el 5 a 1:** es una **hipótesis del grupo**. No es un costo observado del banco ni un dato del dataset. La usamos como penalización relativa provisional y hay que contrastarla con el instructor o con información económica del banco. Acertar vale 0 por simplicidad, aunque detectar un impago no garantiza que se evite la pérdida ni que intervenir sea gratis. Tampoco se midieron las recuperaciones, la eficacia de las acciones ni el costo de atender a todos los clientes marcados.


**Qué hace la celda:** escribe la matriz de penalizaciones relativas. Cada fila es lo que realmente pasó y cada columna lo que dijo el modelo. Acertar vale 0, una falsa alarma vale 1 y no detectar a quien incumple vale 5. Si cambia el `5` y ejecuta las celdas de código, las líneas base se recalculan; los textos y la tabla exploratoria de la sección 4 no.

In [1]:
import numpy as np
import pandas as pd

CLASSES = ["paga", "incumple"]

# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Diagonal = 0.
# SUPUESTO: no detectar un "incumple" cuesta 5 veces más que una falsa alarma.
COST_MATRIX = [[0, 1], [5, 0]]

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
display(cost)

,predice paga,predice incumple
real paga,0,1
real incumple,5,0


## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada; necesita un punto de referencia. Calculamos el costo de dos estrategias sin modelo:
- Decir siempre lo más frecuente («paga»).
- Elegir al azar respetando las proporciones.

Las proporciones vienen de los 30.000 clientes: 77,88 % paga y 22,12 % incumple.


**Qué hace la celda:** toma esas dos proporciones y calcula cuánto costaría en promedio por cliente cada estrategia sin modelo, con la matriz de arriba. Menor es mejor.

In [2]:
# Proporciones en el mismo orden que CLASSES: 23.364 pagan y 6.636 incumplen, de 30.000 clientes.
priors = np.array([23364, 6636]) / 30000

cm = np.array(COST_MATRIX, dtype=float)
most_frequent = int(priors.argmax())
always_most_frequent = priors @ cm[:, most_frequent]
random_by_priors = priors @ cm @ priors
always_flag = priors @ cm[:, 1]
print(f"Siempre '{CLASSES[most_frequent]}':  costo esperado = {always_most_frequent:.2f}")
print(f"Azar según proporciones:   costo esperado = {random_by_priors:.2f}")
print(f"(Referencia) Marcar a todos como 'incumple': costo esperado = {always_flag:.2f}")

Siempre 'paga':  costo esperado = 1.11
Azar según proporciones:   costo esperado = 1.03
(Referencia) Marcar a todos como 'incumple': costo esperado = 0.78


**Lo que dice la línea base:** un modelo que dijera siempre «paga» acertaría el 77,9 % de las veces, pero no detectaría a ningún cliente que incumple. Con la penalización 5 a 1, su costo relativo (1,11) es peor que el del azar (1,03). Marcar a **todos** como «incumple» cuesta menos (0,78), porque no tiene falsos negativos. Es la mejor de las tres estrategias sin modelo, aunque en la práctica significaría revisar a todo el mundo. Estas cifras no son dinero ni ahorro demostrado. Una evaluación seria debe comparar todas las alternativas con las mismas observaciones y tener en cuenta a cuántos clientes puede revisar el equipo.

**¿Sirve la exactitud (accuracy) como métrica aquí?** No. Con 77,9 % de clientes que pagan, un modelo que no detecta a nadie ya tiene 77,9 % de exactitud. Además, la exactitud trata igual los dos errores aunque no cuesten lo mismo.

**Otra línea base** (sección 6 del notebook 02): una regla que marca a quien ya tiene 2 o más meses de atraso tiene una penalización media de 0,775 en el dataset completo, casi igual a marcar a todos. Es una referencia exploratoria. Al comparar modelos hay que usar las mismas observaciones de validación y la misma matriz.


## 4. Cómo se medirá el éxito

- **Métrica principal: ROC AUC.** Mide qué tan bien el modelo ordena a los clientes de menor a mayor riesgo, sin fijar todavía un umbral. Dicho de otra forma: si se toma al azar un cliente que incumple y otro que no, el AUC es la probabilidad de que el modelo le dé más riesgo al que incumple. 0,5 es azar y 1,0 es perfecto. Se eligió porque las clases están desbalanceadas y los datos no traen costos reales para fijar un umbral.
- **Métricas de control** (miran el error que más cuesta):

| Métrica | Qué responde | Por qué importa aquí |
|---|---|---|
| **Recall** de «incumple» | De los que realmente incumplen, ¿a cuántos detectamos? | Es el error caro (falso negativo) |
| **Precision** de «incumple» | De los que marcamos, ¿cuántos realmente incumplen? | Muestra cuántas falsas alarmas genera el equipo |
| **F1** | Equilibrio entre recall y precision | Resumen cuando ambos importan |
| **AUC-PR** | Calidad del ranking centrada en la clase rara | Con 22 % de incumplimiento es más exigente que el AUC. Al azar vale 0,22 |
| **Costo relativo por cliente** | Con la matriz asumida, ¿cuál es la penalización media por equivocarse? | Permite comparar decisiones; no mide dinero ni ahorro real |
| ~~Exactitud~~ | No usarla | «Siempre paga» ya acierta 77,9 % sin detectar a nadie |

- **Criterio provisional de utilidad:** que el modelo ordene mejor que el azar y baje la penalización relativa frente a las líneas base, con las mismas observaciones de validación e informando la variación y el porcentaje de clientes marcados. Eso no basta para probar impacto económico; la meta operativa hay que acordarla con el instructor.

**El umbral teórico con 5 a 1.** Si `p` es la probabilidad calibrada de incumplimiento, decidir «paga» cuesta en promedio `5p` y decidir «incumple» cuesta `1 - p`. Valen lo mismo cuando `p = 1/6 ≈ 0,167`: por encima conviene marcar al cliente y por debajo no. Esto solo vale si las probabilidades del modelo están bien calibradas para la población donde se use, si la matriz `[[0, 1], [5, 0]]` es razonable y si no hay otros costos. No demuestra que el modelo actual esté calibrado ni fija el umbral definitivo.

**El umbral que se usará.** Se elegirá más adelante con validación dentro del entrenamiento, con criterios definidos de antemano, comprobando la calibración del modelo concreto y comparándola con la referencia teórica. La prueba original no se usará para elegirlo.

**Resultados previos que se conservan (exploratorios).** La tabla siguiente ya estaba en este notebook, atribuida al Gradient Boosting optimizado. No hay un procedimiento reproducible que explique cómo se obtuvieron sus umbrales, métricas y costos, así que quedan pendientes de verificar. No afirmamos que los umbrales se hayan elegido bien por validación cruzada.

| Costo de no detectar vs falsa alarma | Umbral reportado (exploratorio) | Recall | Precision | Clientes marcados | Costo relativo modelo | Siempre «paga» | Marcar a todos |
|---|---|---|---|---|---|---|---|
| 2 a 1 | 0,35 | 48,5 % | 57,9 % | 18,6 % | **0,31** | 0,44 | 0,78 |
| **5 a 1** (supuesto) | 0,17 | 70,2 % | 39,3 % | 39,5 % | **0,57** | 1,11 | 0,78 |
| 10 a 1 | 0,09 | 93,2 % | 27,5 % | 75,0 % | **0,69** | 2,21 | 0,78 |

Según estas cifras, el modelo tiene menos penalización relativa que las líneas base en los tres escenarios, y cuanto más se castiga el falso negativo, a más clientes hay que revisar. No son un ahorro demostrado. El ROC AUC **0,7784** viene de una evaluación previa sobre la prueba original, y el AUC-PR **0,5545** (al azar sería 0,2212) también es exploratorio. No se recalcularon aquí.

**Estado de la evaluación.** La prueba original ya se exploró en el EDA con el dataset completo y se evaluó en el notebook de selección, así que no sirve como evaluación final independiente. De aquí en adelante, las decisiones sobre variables, modelo, hiperparámetros, calibración y umbral se toman solo con validación dentro del entrenamiento. Se propone una validación cruzada anidada para evaluar ese procedimiento: reduce el optimismo, pero no borra que los datos ya se vieron. La prueba original se reportará como evaluación complementaria, con esta limitación. Una evaluación realmente independiente necesitaría datos nuevos.


## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?).** Las columnas `PAY_*`, `BILL_AMT*` y `PAY_AMT*` corresponden a abril–septiembre, antes del mes que se predice, así que no se ve una fuga temporal evidente. El riesgo está más bien en el proceso: el EDA ya usó el dataset completo y la prueba original ya se evaluó. En adelante, todo ajuste y toda selección se hacen dentro del entrenamiento, y esa prueba no se presenta como independiente.
- **Sesgos y equidad (¿a quién podría perjudicar el modelo?).** `SEX`, `AGE` y `MARRIAGE` son atributos sensibles y el modelo podría perjudicar a un grupo. Se compararán los errores por grupo en la etapa de validación.
- **Datos personales.** No hay nombres ni documentos, solo un `ID` anónimo que se excluye del modelo. La licencia del dataset es CC BY 4.0 (verificada en UCI) y permite redistribuirlo citando a Yeh (2009).
- **Hasta dónde decide el modelo.** Ordena por riesgo y ayuda a priorizar revisiones; no aprueba ni niega crédito. La decisión final sobre un cliente la toma una persona del área de riesgo. Además, los datos son de un solo banco de Taiwán en 2005 y pueden no servir en otro país o época.


## 5b. ¿Qué tipo de modelo tiene sentido para este caso?
Lo que pide el negocio decide el tipo de modelo:

| El negocio necesita… | Entonces el modelo debe… |
|---|---|
| **Detectar al cliente que va a incumplir** (el error caro) | Ordenar muy bien por riesgo (ROC AUC alto) |
| **Decidir un umbral según costos** | Dar probabilidades confiables (bien calibradas), no solo «sí/no» |
| **Justificar por qué se marcó a un cliente** | Poder explicarse (SHAP, o un modelo simple como contraste) |

**Recomendación con la evidencia actual:**
- **Candidato principal: gradient boosting** (Gradient Boosting, LightGBM o similar). En la exploración previa, la validación cruzada da ≈ 0,78 de ROC AUC frente a 0,73 de la regresión logística. El notebook 02 muestra buenos indicios de calibración para HistGradientBoosting, pero no es una garantía para toda la familia ni para el Gradient Boosting optimizado; hay que comprobarlo en el modelo concreto antes de aplicar el umbral de costos.
- **Modelo de contraste: regresión logística.** Es menos precisa pero fácil de explicar. Sirve para comprobar que la complejidad del boosting vale la pena.
- **Explicación:** SHAP sobre el modelo principal, para decir qué variables llevaron a marcar a un cliente.

**Limitación:** hasta ahora solo se optimizó el Gradient Boosting; los demás corrieron con valores por defecto. La decisión final se toma en el torneo (`05_torneo.ipynb`), con la misma métrica, los mismos pliegues y el ajuste de cada candidato.


## 6. Cierre

- [x] Pregunta, quién decide y costos escritos con números (el 5 a 1 es una hipótesis por validar con el instructor).
- [x] Línea base calculada (y una regla simple más exigente en el notebook 02).
- [x] Métrica principal elegida y justificada, con métricas de control.
- [x] Tipo de modelo recomendado y su justificación.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que no escribió este notebook sabe explicar por qué un error cuesta más que el otro.
